<a href="https://colab.research.google.com/github/kbrezinski/Chaos/blob/main/test_dropout_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytorch-lightning==2.0.7 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 19.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

import pytorch_lightning as pl
from torchmetrics import Accuracy  # Import Accuracy metric

In [ ]:
import torchvision.models as models

# Define Logistic Map Dropout
class LogisticMapDropout(nn.Module):
    def __init__(self, lambda_val=3.9, epsilon=0.5):
        super(LogisticMapDropout, self).__init__()
        self.lambda_val = lambda_val
        self.epsilon = epsilon
        self.register_buffer('state', None)  # Register `state` as a non-trainable buffer

    def forward(self, x):
        if self.training:

            # Initialize state lazily
            if self.state is None:
                self.state = torch.rand_like(x, requires_grad=False).to(x.device)

            # Update logistic map state
            self.state = self.lambda_val * self.state * (1 - self.state)

            # Apply Sparse Scaling
            scaling = torch.where(self.state <= self.epsilon, 1.0, self.state)

            return x * scaling
        else:
            return x * 0.5

    def _calculate_test_time_scaling(self):
        # Calculate the expected scaling factor during training
        # P(s >= epsilon) and P(s < epsilon)
        p_preserve = (1 - self.epsilon)  # Probability s >= epsilon (approximation)
        p_scale = self.epsilon          # Probability s < epsilon

        # Expected value of s when s < epsilon (assume uniform distribution for simplicity)
        avg_scaled = self.epsilon / 2  # Mean of uniform(0, epsilon)

        # Combine probabilities and expected values
        return p_preserve * 1.0 + p_scale * avg_scaled


class BasicBlockWithDropout(models.resnet.BasicBlock):
    def __init__(self, in_planes, out_planes, stride=1, padding=1, downsample=None):
        super(BasicBlockWithDropout, self).__init__(in_planes,out_planes, stride, downsample)
        self.dropout = LogisticMapDropout(lambda_val=3.9)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)

        # Add a downsample layer if the dimensions do not match
        self.downsample = None
        if stride != 1 or in_planes != out_planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_planes),
            )

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


In [ ]:
from torch import optim

class ResNetWithLogisticMapDropout(pl.LightningModule):
    def __init__(self, resnet_type, num_classes=10, learning_rate=0.1,
                 momentum=0.9, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters()  # Save hyperparameters automatically
        self.resnet = resnet_type(num_classes=num_classes)

        self.criterion = nn.CrossEntropyLoss()  # Move loss function here
        self.val_accuracy = Accuracy(task='multiclass', num_classes=num_classes)
        self.resnet.layer1 = nn.Sequential(
            BasicBlockWithDropout(64, 64),
            BasicBlockWithDropout(64, 64)
            )
        self.resnet.layer2 = nn.Sequential(
            BasicBlockWithDropout(64, 128, stride=2),
            BasicBlockWithDropout(128, 128)
        )
        self.resnet.layer3 = nn.Sequential(
            BasicBlockWithDropout(128, 256, stride=2),
            BasicBlockWithDropout(256, 256)
        )
        self.resnet.layer4 = nn.Sequential(
            BasicBlockWithDropout(256, 512, stride=2),
            BasicBlockWithDropout(512, 512)
        )

    def forward(self, x):
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)

        x = self.resnet.layer1(x)
        x = self.resnet.layer2(x)
        x = self.resnet.layer3(x)
        x = self.resnet.layer4(x)

        x = self.resnet.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        self.log("train_loss", loss)  # Log training loss
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        _, predicted = torch.max(outputs.data, 1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
        val_error = 100 * (1 - (correct / total))  # Calculate validation % error
        self.log("val_error", val_error, prog_bar=True)  # Log only validation % error

    def on_train_epoch_end(self):
        self.logged_metrics = {  # Store logged metrics in a dictionary
            "train_loss": self.trainer.callback_metrics["train_loss"].item(),
            "val_error": self.trainer.callback_metrics["val_error"].item(),
            # Add other metrics you want to save
        }

    def configure_optimizers(self):
        optimizer = optim.SGD(self.parameters(), lr=self.hparams.learning_rate,
                              momentum=self.hparams.momentum, weight_decay=self.hparams.weight_decay)
        return optimizer

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger

class SaveLoggedMetricsCallback(Callback):
    def on_save_checkpoint(self, trainer, pl_module, checkpoint):
        # Access logged_metrics from your LightningModule:
        checkpoint['logged_metrics'] = pl_module.logged_metrics

checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",  # Directory to save checkpoints
    filename="resnet-logistic-{epoch:02d}-{val_error:.2f}",  # Checkpoint filename format
    save_top_k=1,  # Save the top 3 checkpoints based on validation error
    monitor="val_error",  # Metric to monitor for checkpointing
    mode="min",  # Save checkpoints when the monitored metric is minimized
    every_n_epochs=5 # Save every k epochs (replace k with your desired value)
)

In [ ]:
import os
import glob
from torchvision.models import resnet18

# Data loading and preprocessing
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=1024, shuffle=True, num_workers=os.cpu_count(), pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=1024, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)

csv_logger = CSVLogger(save_dir="logs/", name="my_experiment", version='0')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetWithLogisticMapDropout(resnet_type=resnet18).to(device)

# Find the most recent checkpoint file
checkpoint_dir = "checkpoints"  # Your checkpoint directory
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "*.ckpt"))
latest_checkpoint = max(checkpoint_files, key=os.path.getctime) if checkpoint_files else None

# Create and train the model
trainer = pl.Trainer(
    max_epochs=100,
    log_every_n_steps=10,
    default_root_dir="logs",
    logger=csv_logger,
    accelerator='gpu',
    devices=1 if torch.cuda.is_available() else None,
    precision=16,
    accumulate_grad_batches=2,
    callbacks=[checkpoint_callback, SaveLoggedMetricsCallback()])

trainer.fit(model, trainloader, testloader, ckpt_path=latest_checkpoint)

Files already downloaded and verified
Files already downloaded and verified


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type               | Params
----------------------------------------------------
0 | resnet       | ResNet             | 11.2 M
1 | criterion    | CrossEntropyLoss   | 0     
2 | val_accuracy | MulticlassAccuracy | 0     
----------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.
